# 02 — Data Understanding


In [2]:
import sys
sys.path.insert(0, '..')

import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

from src.data.load_data import load_csv
from src.data.data_quality import data_quality_summary
from src.analysis.descriptive_analysis import describe_numeric, summary_by_defect_status, value_distribution
from src.utils.config import load_config, resolve_path

config = load_config()
df = load_csv(resolve_path(config['data']['raw_path']))
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")

2026-09-21 00:02:25 | INFO     | src.data.load_data | Loaded CSV 'manufacturing_defect_dataset.csv' with shape (3240, 17)


Loaded 3240 rows, 17 columns


## Descriptive statistics for every numeric column



In [3]:
describe_numeric(df)

,count,mean,std,min,25%,50%,75%,max
ProductionVolume,3240.0,548.523148,262.402073,100.000000,322.000000,549.000000,775.250000,999.000000
ProductionCost,3240.0,12423.018476,4308.051904,5000.174521,8728.829280,12405.204656,16124.462428,19993.365549
SupplierQuality,3240.0,89.833290,5.759143,80.004820,84.869219,89.704861,94.789936,99.989214
DeliveryDelay,3240.0,2.558951,1.705804,0.000000,1.000000,3.000000,4.000000,5.000000
DefectRate,3240.0,2.749116,1.310154,0.500710,1.598033,2.708775,3.904533,4.998529
QualityScore,3240.0,80.134272,11.611750,60.010098,70.103420,80.265312,90.353822,99.996993
MaintenanceHours,3240.0,11.476543,6.872684,0.000000,5.750000,12.000000,17.000000,23.000000
DowntimePercentage,3240.0,2.501373,1.443684,0.001665,1.264597,2.465151,3.774861,4.997591
InventoryTurnover,3240.0,6.019662,2.329791,2.001611,3.983249,6.022389,8.050222,9.998577
StockoutRate,3240.0,0.050878,0.028797,0.000002,0.026200,0.051837,0.075473,0.099997


**Interpretation:**
- `ProductionVolume` ranges from 100 to 999 units, averaging ~549.
- `ProductionCost` ranges from about $5,000 to $20,000, averaging ~$12,423.
- `QualityScore` and `SupplierQuality` both sit on roughly a 0–100 (actually ~60–100 and ~80–100 respectively) scale, consistent with being quality/score metrics.
- `DefectRate` ranges from about 0.5 to 5.0 — a continuous defect measurement distinct from the binary `DefectStatus` flag.
- Several columns (`DowntimePercentage`, `StockoutRate`, `EnergyEfficiency`) are small decimal rates, as their names suggest.

## Target variable: DefectStatus



In [4]:
counts = df['DefectStatus'].value_counts().sort_index()
pct = (counts / len(df) * 100).round(2)
summary = pd.DataFrame({'count': counts, 'percentage': pct})
summary.index = summary.index.map({0: 'Not Defective', 1: 'Defective'})
summary

,count,percentage
DefectStatus,,
Not Defective,517,15.96
Defective,2723,84.04


**Interpretation:** The dataset is notably **imbalanced**: 84.0% of records are flagged as defective, and only 16.0% are not. This is an important limitation to keep in mind — it means the majority-class default (assuming 'defective') would already be correct 84% of the time, so any variable's *relative* association with the outcome matters more than its raw correlation coefficient alone.

## Discrete/low-cardinality columns



In [5]:
value_distribution(df, 'DeliveryDelay')

,count,percentage
DeliveryDelay,,
0,521,16.08
1,516,15.93
2,512,15.80
3,569,17.56
4,566,17.47
5,556,17.16


In [6]:
value_distribution(df, 'SafetyIncidents')

,count,percentage
SafetyIncidents,,
0,293,9.04
1,345,10.65
2,304,9.38
3,342,10.56
4,307,9.48
5,303,9.35
6,333,10.28
7,317,9.78
8,326,10.06


**Interpretation:** `DeliveryDelay` takes integer values from 0 to 5, roughly evenly distributed. `SafetyIncidents` takes integer values from 0 to 9, also roughly evenly spread — there is no single dominant safety-incident count.

## Averages split by defect status


In [7]:
summary_by_defect_status(df)

DefectStatus,0,1
ProductionVolume,470.866538,563.267352
ProductionCost,12158.877326,12473.169403
SupplierQuality,89.328686,89.929096
DeliveryDelay,2.537718,2.562982
DefectRate,2.010328,2.889386
QualityScore,85.442375,79.126454
MaintenanceHours,6.791103,12.366140
DowntimePercentage,2.487697,2.503970
InventoryTurnover,5.983667,6.026496
StockoutRate,0.048197,0.051387


**Interpretation (preliminary — full analysis in notebook 06):** A few variables show a visibly different average between defective and non-defective records — most notably `MaintenanceHours` (higher for defective records) and `QualityScore` (lower for defective records). This is an early signal that will be tested more rigorously with correlation analysis in the next notebook.

## Dta Quality Summary


In [8]:
data_quality_summary(df)

2026-09-21 00:02:25 | INFO     | src.data.data_quality | Data quality summary — shape=(3240, 17), duplicate_rows=0, columns_with_missing=0


{'shape': (3240, 17),
 'missing_values':                       missing_count  missing_percentage
 ProductionVolume                  0                 0.0
 ProductionCost                    0                 0.0
 SupplierQuality                   0                 0.0
 DeliveryDelay                     0                 0.0
 DefectRate                        0                 0.0
 QualityScore                      0                 0.0
 MaintenanceHours                  0                 0.0
 DowntimePercentage                0                 0.0
 InventoryTurnover                 0                 0.0
 StockoutRate                      0                 0.0
 WorkerProductivity                0                 0.0
 SafetyIncidents                   0                 0.0
 EnergyConsumption                 0                 0.0
 EnergyEfficiency                  0                 0.0
 AdditiveProcessTime               0                 0.0
 AdditiveMaterialCost              0            

## Summary of this notebook

- All 17 columns are numeric and have been described statistically.
- `DefectStatus` is imbalanced: 84.0% defective, 16.0% not defective — an important interpretive caveat for the rest of the project.
- `MaintenanceHours` and `QualityScore` show early, visible differences between defective and non-defective groups, motivating deeper investigation in the EDA and business-analysis notebooks.
- No date column exists, confirming that time-trend analysis remains out of scope.

This data-understanding foundation feeds directly into `docs/data_dictionary.md` and the cleaning decisions made in `notebooks/03_data_cleaning.ipynb`.